# 27: Can Stored Material History Redirect the Future?

This notebook audits the construct-validity story. The point is not to make hidden state into memory. The point is to separate an invalid downstream V1 intervention, a surviving immediate mechanism, a corrected V2 directional effect, and an unresolved minimum-magnitude claim.

In [1]:
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_PROFILE = os.environ.get("NOTEBOOK_PROFILE", "quick")
RUN_CANONICAL = os.environ.get("RUN_CANONICAL", "0") == "1"


def find_repo_root(start=Path.cwd()):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "content" / "books" / "digital-life").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
FIG_DIR = NOTEBOOK_DIR / "generated-figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def require_path(relative_path):
    path = REPO_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def summarize_result(label, result, status=None, source="canonical research artifact"):
    row = {"label": label, "source": source}
    if status is not None:
        row["status"] = status
    for key in ["n", "mean", "ci95_low", "ci95_high", "achieved_mde80_one_sided"]:
        if key in result:
            row[key] = result[key]
    return row

print("profile", NOTEBOOK_PROFILE, "run_canonical", RUN_CANONICAL)
print("repo", REPO_ROOT)

CHAPTER = 27
MANUSCRIPT = require_path("content/books/digital-life/27-can-stored-history-redirect-the-future/index.md")
LINEAGE = [
    "scripts/books/digital-life/ch27_digital_crystal_decaying_material_history_causal_response_v1.py",
    "scripts/books/digital-life/ch27_v1_construct_validity_mechanism_audit.py",
    "scripts/books/digital-life/ch27_digital_crystal_decaying_material_history_causal_response_v2.py",
    "scripts/books/digital-life/ch27_v2_trajectory_closeout_audit.py",
]
for item in LINEAGE:
    require_path(item)
print("manuscript", MANUSCRIPT.relative_to(REPO_ROOT))
print("lineage ok", len(LINEAGE))

profile quick run_canonical False
repo C:\Projects\working-book
manuscript content\books\digital-life\27-can-stored-history-redirect-the-future\index.md
lineage ok 4


## Mechanism: Logistic Saturation

This recomputed toy mechanism shows why positive stored material can reduce incremental causal sensitivity. Raising the baseline probability moves a candidate along the sigmoid; near saturation, the same score increment produces a smaller change in probability.

In [2]:
import numpy as np
score = np.linspace(-6, 6, 300)
delta = 0.7
p0 = 1 / (1 + np.exp(-score))
p1 = 1 / (1 + np.exp(-(score + delta)))
sensitivity = p1 - p0
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(score, p0, label="baseline p")
ax.plot(score, sensitivity, label="incremental Δp for +0.7 score")
ax.set_title("RECOMPUTED DEMO: saturation can reduce causal sensitivity")
ax.set_xlabel("stored-material-shifted score")
ax.legend(frameon=False)
path = FIG_DIR / "ch27-logistic-sensitivity-demo.png"
fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)
path.relative_to(REPO_ROOT)

WindowsPath('notebooks/generated-figures/ch27-logistic-sensitivity-demo.png')

## Canonical Artifact Audit

Loaded values are canonical research artifacts, not recomputed in this notebook run.

In [3]:
v1_verdict = load_json("research/digital-life/ch27-decaying-material-history-causal-response-v1/stage-07-verdict.json")
v1_audit = load_json("research/digital-life/ch27-v1-construct-validity-audit/ch27-v1-construct-validity-audit-report.json")
v2_primary = load_json("research/digital-life/ch27-decaying-material-history-causal-response-v2/stage-04-primary.json")
v2_secondary = load_json("research/digital-life/ch27-decaying-material-history-causal-response-v2/stage-05-secondary.json")
v2_verdict = load_json("research/digital-life/ch27-decaying-material-history-causal-response-v2/stage-07-verdict.json")
closeout = load_json("research/digital-life/ch27-v2-trajectory-closeout-audit/ch27-v2-trajectory-closeout-audit.json")

pd.DataFrame([
    {"claim": "V1 downstream FORCE/PREVENT", "status": v1_audit["metadata"]["v1_primary_construct_validity"], "source": "canonical audit", "role": "INVALID_IMPLEMENTATION"},
    {"claim": "V1 immediate E1 mechanism", "status": v1_audit["metadata"]["v1_immediate_E1_status"], "source": "canonical audit", "role": "surviving sub-result"},
    {"claim": v2_primary["contrast"], "status": v2_primary["status"], "source": "canonical research artifact", "role": "direction supported; magnitude unresolved"},
    {"claim": "trajectory closeout", "status": closeout["metadata"]["formal_V2_primary_status_unchanged"], "source": "descriptive audit", "role": "DESCRIPTIVE_ONLY"},
])

,claim,status,source,role
0,V1 downstream FORCE/PREVENT,INVALID_PREVENT_DID_NOT_EXPLICITLY_BLOCK_X,canonical audit,INVALID_IMPLEMENTATION
1,V1 immediate E1 mechanism,REMAINS_INTERPRETABLE,canonical audit,surviving sub-result
2,RB_G_local(accessible) - RB_G_local(remote),UNRESOLVED,canonical research artifact,direction supported; magnitude unresolved
3,trajectory closeout,UNRESOLVED,descriptive audit,DESCRIPTIVE_ONLY


In [4]:
pd.DataFrame([summarize_result(v2_primary["contrast"], v2_primary["result"], v2_primary["status"])])

,label,source,status,n,mean,ci95_low,ci95_high,achieved_mde80_one_sided
0,RB_G_local(accessible) - RB_G_local(remote),canonical research artifact,UNRESOLVED,192,-0.397245,-0.678592,-0.11922,0.357057


In [5]:
per_lag_rows = []
for lag, values in closeout["per_lag"].items():
    per_lag_rows.append({
        "lag": int(lag),
        "delta_increment": values["delta_RB_increment"]["mean"],
        "cumulative_delta": values["cumulative_delta_RB"]["mean"],
        "material_mass": values.get("material_mass", {}).get("mean", None),
    })
per_lag = pd.DataFrame(per_lag_rows).sort_values("lag")
fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.plot(per_lag["lag"], per_lag["cumulative_delta"], marker="o", label="cumulative RB consequence")
if per_lag["material_mass"].notna().any():
    ax2 = ax.twinx(); ax2.plot(per_lag["lag"], per_lag["material_mass"], color="#c46a2b", marker="s", label="material mass")
    ax2.set_ylabel("material mass")
ax.axhline(0, color="#333333", lw=1)
ax.set_title("CANONICAL ARTIFACT: decaying trace vs accumulating consequence")
ax.set_xlabel("lag"); ax.set_ylabel("cumulative accessible - remote")
path = FIG_DIR / "ch27-decay-vs-consequence.png"
fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)
print(path.relative_to(REPO_ROOT))
per_lag.head(), per_lag.tail()

notebooks\generated-figures\ch27-decay-vs-consequence.png


(   lag  delta_increment  cumulative_delta material_mass
 0    1        -0.014985         -0.014985          None
 1    2        -0.052980         -0.067965          None
 2    3        -0.031493         -0.099458          None
 3    4        -0.020620         -0.120077          None
 4    5        -0.027016         -0.147093          None,
     lag  delta_increment  cumulative_delta material_mass
 7     8        -0.021971         -0.277838          None
 8     9        -0.046887         -0.324725          None
 9    10        -0.046442         -0.371167          None
 10   11        -0.017776         -0.388943          None
 11   12        -0.008303         -0.397245          None)

## Result Ledger

- **Invalid:** V1 downstream twelve-step primary for the intended intervention, because PREVENT did not explicitly block `x` during lag 1.
- **Supported lower-level mechanism:** accessible hidden material changes immediate local sensitivity through the logistic response.
- **V2 result:** downstream Rao-Blackwellized accessible-minus-remote direction is negative and its CI excludes zero, but the achieved MDE is larger than the frozen ±0.15 minimum-magnitude requirement.
- **Closeout:** descriptive trajectory persistence does not rescue the unresolved confirmatory magnitude claim.

`HIDDEN STATE != MEMORY` and `DESCRIPTIVE CLOSEOUT != CONFIRMATORY RESCUE`.